# Costruzione decrizione automatica svg circuiti

## Esempio estrazione

In [55]:
import xml.etree.ElementTree as ET
import re

def apri_file_svg(nome_file):
    tree = ET.parse(nome_file)
    root = tree.getroot()
    return root

# Esempio di utilizzo
nome_file_svg = 'circuito.svg'
root_svg = apri_file_svg(nome_file_svg)

#find element with id="circuito1"
for child in root_svg:
    if child.attrib['id'] == 'circuito1':
        circuito1 = child

#foreach element in circuito1 print id
for child in circuito1:
    if not re.search('title', child.attrib['id']):
        print('RAMO: ', child.attrib['id'])

        #forreach element in child print id
        for child2 in child:
            if not re.search('title', child2.attrib['id']):
                print('- ', child2.attrib['id'])
                
                #get text by id valore-r6
                for child3 in child2:
                    if re.search('valore', child3.attrib['id']):
                        for child4 in child3:
                            print('---- ', child4.text)


    print('===============================')

RAMO:  ramo-ga
-  nodo-a
-  link1-ga
-  r6
----  3 Ω
-  link2-ga
RAMO:  ramo-fg
-  link1-fg
RAMO:  ramo-hg
-  nodo-g
-  link2-hg
-  r5
----  4 Ω
-  link1-hg
RAMO:  ramo-ef
-  nodo-f
-  link2-ef
-  r4
----  6 Ω
-  link1-ef
RAMO:  ramo-eh
-  link2-eh
-  g2
----  6 V
-  link1-eh
RAMO:  ramo-ch
-  nodo-h
-  link2-ch
-  r2
----  6 Ω
-  link1-ch
RAMO:  ramo-de
-  nodo-e
-  link1-de
RAMO:  ramo-cd
-  nodo-d
-  link2-bc
-  r3
----  4 Ω
-  link-bc
RAMO:  ramo-bc
-  nodo-c
-  link1-bc
RAMO:  ramo-ab
-  nodo-b
-  link3-ab
-  r1
----  3 Ω
-  link2-ab
-  g1
----  42 V
-  link1-ab


## Gestione delle coppie presenti nod1-nodo2

### Estrazione

In [56]:
import xml.etree.ElementTree as ET
import re

def apri_file_svg(nome_file):
    tree = ET.parse(nome_file)
    root = tree.getroot()
    return root

# Esempio di utilizzo
nome_file_svg = 'circuito.svg'
root_svg = apri_file_svg(nome_file_svg)

# Trova l'elemento con id="circuito1"
for child in root_svg:
    if child.attrib['id'] == 'circuito1':
        circuito1 = child

        # Inizializza un dizionario per i rami
rami = {}

valori_componenti = {}

for child in circuito1:
    if not re.search('title', child.attrib['id']):
        nome_ramo = child.attrib['id'].replace('ramo-', '')
        nodo_iniziale = nome_ramo[0].upper()
        nodo_finale = nome_ramo[1].upper()

        nome_ramo = nodo_iniziale + '-' + nodo_finale
        rami[nome_ramo] = []

        # Itera sugli elementi figlio e aggiungi gli id al dizionario
        for child2 in child:
            if not re.search('title', child2.attrib['id']):
                #pop into the list
                rami[nome_ramo].append(child2.attrib['id'])

                #get text by id valore-r6
                for child3 in child2:
                    if re.search('valore', child3.attrib['id']):
                        for child4 in child3:
                            valori_componenti[child2.attrib['id']] = child4.text

        #reverse the list
        rami[nome_ramo].reverse()

        

#reverse the dictionary
rami = dict(reversed(list(rami.items())))
rami, valori_componenti

({'A-B': ['link1-ab', 'g1', 'link2-ab', 'r1', 'link3-ab', 'nodo-b'],
  'B-C': ['link1-bc', 'nodo-c'],
  'C-D': ['link-bc', 'r3', 'link2-bc', 'nodo-d'],
  'D-E': ['link1-de', 'nodo-e'],
  'C-H': ['link1-ch', 'r2', 'link2-ch', 'nodo-h'],
  'E-H': ['link1-eh', 'g2', 'link2-eh'],
  'E-F': ['link1-ef', 'r4', 'link2-ef', 'nodo-f'],
  'H-G': ['link1-hg', 'r5', 'link2-hg', 'nodo-g'],
  'F-G': ['link1-fg'],
  'G-A': ['link2-ga', 'r6', 'link1-ga', 'nodo-a']},
 {'r6': '3 Ω',
  'r5': '4 Ω',
  'r4': '6 Ω',
  'g2': '6 V',
  'r2': '6 Ω',
  'r3': '4 Ω',
  'r1': '3 Ω',
  'g1': '42 V'})

### Generazione

In [57]:
risposte = {}
for ramo in rami:
    domande = [
        '* ' + ramo[0] + ' * ' + ramo[2] + ' *',
    ]

    text = 'Il ramo che va da ' + ramo[0] + ' a ' + ramo[2] + ' è composto da '
    
    for elemento in rami[ramo]:
        if not re.search('link', elemento) and not re.search('nodo', elemento):
            # check if elemento has r + some number
            if re.search('r\d+', elemento):
                text += 'una resistenza ' + elemento.upper() + ' seguita da '
            elif re.search('g\d+', elemento):
                text += 'un generatore seguito da '

    #if the last substring is 'seguito da' remove it
    if text.endswith('seguito da '):
        text = text[:-12]

    if text.endswith('seguita da '):
        text = text[:-12]

    if text.endswith('è composto da '):
        text = text[:-15]
        text += ' non è composto da alcun elemento'

    text += '<image>circuiti/circuito.svg</image>'
    for elemento in rami[ramo]:
        if not re.search('link', elemento) and not re.search('nodo', elemento):
            text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '</svgElement>'
    

    for domanda in domande:
        domanda_uppercase = domanda.upper()
        risposte[domanda_uppercase] = text

## Gestione e inferenza di nuove coppie (es. da nodo2 a nodo7)

## Estrazione

In [58]:
import xml.etree.ElementTree as ET
import re

def apri_file_svg(nome_file):
    tree = ET.parse(nome_file)
    root = tree.getroot()
    return root

# Esempio di utilizzo
nome_file_svg = 'circuito.svg'
root_svg = apri_file_svg(nome_file_svg)

# Trova l'elemento con id="circuito1"
for child in root_svg:
    if child.attrib['id'] == 'circuito1':
        circuito1 = child

        # Inizializza un dizionario per i rami
rami = {}
valori_componenti = {}
for child in circuito1:
    if not re.search('title', child.attrib['id']):
        nome_ramo = child.attrib['id'].replace('ramo-', '')
        nodo_iniziale = nome_ramo[0].upper()
        nodo_finale = nome_ramo[1].upper()

        nome_ramo = nodo_iniziale + '-' + nodo_finale
        rami[nome_ramo] = []

        # Itera sugli elementi figlio e aggiungi gli id al dizionario
        for child2 in child:
            if not re.search('title', child2.attrib['id']):
                #pop into the list
                rami[nome_ramo].append(child2.attrib['id'])

                #get text by id valore-r6
                for child3 in child2:
                    if re.search('valore', child3.attrib['id']):
                        for child4 in child3:
                            valori_componenti[child2.attrib['id']] = child4.text

        #reverse the list
        rami[nome_ramo].reverse()

#reverse the dictionary
rami = dict(reversed(list(rami.items())))


risposte_complete = {}

for ramo in rami:
    nodo_iniziale = ramo[0]
    nodo_finale = ramo[2]

    text = rami[ramo]

    risposte_complete[nodo_iniziale + '-' + nodo_finale] = text

    for ramo2 in rami:
        if not ramo2[2] == nodo_iniziale:
            if ramo2[0] == nodo_finale and ramo2 != ramo:
                text = text + rami[ramo2]
                risposte_complete[nodo_iniziale + '-' + ramo2[2]] = text
                nodo_finale = ramo2[2]
        else:
            break

rami, valori_componenti


({'A-B': ['link1-ab', 'g1', 'link2-ab', 'r1', 'link3-ab', 'nodo-b'],
  'B-C': ['link1-bc', 'nodo-c'],
  'C-D': ['link-bc', 'r3', 'link2-bc', 'nodo-d'],
  'D-E': ['link1-de', 'nodo-e'],
  'C-H': ['link1-ch', 'r2', 'link2-ch', 'nodo-h'],
  'E-H': ['link1-eh', 'g2', 'link2-eh'],
  'E-F': ['link1-ef', 'r4', 'link2-ef', 'nodo-f'],
  'H-G': ['link1-hg', 'r5', 'link2-hg', 'nodo-g'],
  'F-G': ['link1-fg'],
  'G-A': ['link2-ga', 'r6', 'link1-ga', 'nodo-a']},
 {'r6': '3 Ω',
  'r5': '4 Ω',
  'r4': '6 Ω',
  'g2': '6 V',
  'r2': '6 Ω',
  'r3': '4 Ω',
  'r1': '3 Ω',
  'g1': '42 V'})

### Generazione

In [59]:
risposte = {}

- Domanda: parlami del circuito di esercizio
- Risposta: il circuito ...

In [60]:
domande = [
    '* circuito * esercizio *',
    '* circuito * generale *',
]

# numero nodi
numNodi = 0
nodi = []
for ramo in rami:
    if not ramo[0] in nodi:
        nodi.append(ramo[0])
        numNodi += 1
    if not ramo[2] in nodi:
        nodi.append(ramo[2])
        numNodi += 1

# elementi per ramo
elementiPerRamo = []
for ramo in rami:
    sentence = 'Il ramo che va da ' + ramo[0] + ' a ' + ramo[2] + ' è composto da '
    
    for elemento in rami[ramo]:
        if not re.search('link', elemento) and not re.search('nodo', elemento):
            # check if elemento has r + some number
            if re.search('r\d+', elemento):
                sentence += 'una resistenza ' + elemento.upper() + ' seguita da '
            elif re.search('g\d+', elemento):
                sentence += 'un generatore seguito da '

    #if the last substring is 'seguito da' remove it
    if sentence.endswith('seguito da '):
        sentence = sentence[:-12]

    if sentence.endswith('seguita da '):
        sentence = sentence[:-12]

    if sentence.endswith('è composto da '):
        sentence = sentence[:-15]
        sentence += ' non è composto da alcun elemento'

    elementiPerRamo.append(sentence)

text = 'Il circuito è composto da ' + str(numNodi) + ' nodi e da ' + str(len(rami)) + ' rami. '
text += '. '.join(elementiPerRamo)
text += '<image>circuiti/circuito.svg</image>'

# risposte
for domanda in domande:
    domanda_generale = domanda.upper()
    risposte[domanda_generale] = text
    print(domanda_generale, risposte[domanda_generale])

* CIRCUITO * ESERCIZIO * Il circuito è composto da 8 nodi e da 10 rami. Il ramo che va da A a B è composto da un generatore seguito da una resistenza R1. Il ramo che va da B a C non è composto da alcun elemento. Il ramo che va da C a D è composto da una resistenza R3. Il ramo che va da D a E non è composto da alcun elemento. Il ramo che va da C a H è composto da una resistenza R2. Il ramo che va da E a H è composto da un generatore. Il ramo che va da E a F è composto da una resistenza R4. Il ramo che va da H a G è composto da una resistenza R5. Il ramo che va da F a G non è composto da alcun elemento. Il ramo che va da G a A è composto da una resistenza R6<image>circuiti/circuito.svg</image>
* CIRCUITO * GENERALE * Il circuito è composto da 8 nodi e da 10 rami. Il ramo che va da A a B è composto da un generatore seguito da una resistenza R1. Il ramo che va da B a C non è composto da alcun elemento. Il ramo che va da C a D è composto da una resistenza R3. Il ramo che va da D a E non è c

- Domanda:     c'è una resistenza/generatore tra il nodo X e il nodo Y?
- Risposta:    si/no, ...

In [61]:
for ramo in risposte_complete:
    domande_resistenze = [
        '* resistenza * ' + ramo[0] + ' * ' + ramo[2] + ' *',
        '* resistenza * ' + ramo[2] + ' * ' + ramo[0] + ' *',
    ]

    domande_generatori = [
        '* generatore * ' + ramo[0] + ' * ' + ramo[2] + ' *',
        '* generatore * ' + ramo[2] + ' * ' + ramo[0] + ' *',
    ]

    text_resistenza = ''
    text_generatore = ''

    count_generatori = 0
    
    for elemento in risposte_complete[ramo]:
        if not re.search('link', elemento) and not re.search('nodo', elemento):
            if re.search('r\d+', elemento):
                if text_resistenza == '':
                    text_resistenza += 'si, c\'è la resistenza ' + elemento.upper()
                else:
                    text_resistenza += 'seguita dalla resistenza ' + elemento.upper()
            elif re.search('g\d+', elemento):
                count_generatori += 1

    if text_resistenza == '':
        text_resistenza = 'no, non c\'è alcuna resistenza'

    if count_generatori > 0:
        if count_generatori == 1:
            text_generatore = 'si, c\'è un generatore'
        else:
            text_generatore = 'si, ci sono ' + str(count_generatori) + ' generatori'
    else:
        text_generatore = 'no, non c\'è alcun generatore'

    text_resistenza += '<image>circuiti/circuito.svg</image>'
    text_generatore += '<image>circuiti/circuito.svg</image>'

    for elemento in risposte_complete[ramo]:
        if not re.search('link', elemento) and not re.search('nodo', elemento):
            # check if elemento has r + some number
            if re.search('r\d+', elemento):
                text_resistenza += '<svgElement style-name="stroke" style-value="#04ed00">simbolo-' + elemento + '</svgElement>'
            elif re.search('g\d+', elemento):
                text_generatore += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-1</svgElement>'
                text_generatore += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-2</svgElement>'
                text_generatore += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-3</svgElement>'
                text_generatore += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-4</svgElement>'


    for domanda_resistenza in domande_resistenze:
        domanda_resistenza_uppercase = domanda_resistenza.upper()
        risposte[domanda_resistenza_uppercase] = text_resistenza

    for domanda_generatore in domande_generatori:
        domanda_generatore_uppercase = domanda_generatore.upper()
        risposte[domanda_generatore_uppercase] = text_generatore

- Domanda: cosa c'è tra il nodo X e il nodo Y?
- Risposta: il ramo che va ... è composto da ...

In [62]:
for ramo in risposte_complete:
    domande = [
        '* nodo ' + ramo[0] + ' * nodo ' + ramo[2] + ' *',
        '* nodo ' + ramo[2] + ' * nodo ' + ramo[0] + ' *',
    ]

    text = 'Il ramo che va da ' + ramo[0] + ' a ' + ramo[2] + ' è composto da '
    
    for elemento in risposte_complete[ramo]:
        if not re.search('link', elemento) and not re.search('nodo', elemento):
            # check if elemento has r + some number
            if re.search('r\d+', elemento):
                text += 'una resistenza ' + elemento.upper() + ' seguita da '
            elif re.search('g\d+', elemento):
                text += 'un generatore seguito da '

    #if the last substring is 'seguito da' remove it
    if text.endswith('seguito da '):
        text = text[:-12]

    if text.endswith('seguita da '):
        text = text[:-12]

    if text.endswith('è composto da '):
        text = text[:-15]
        text += ' non è composto da alcun elemento'

    text += '<image>circuiti/circuito.svg</image>'
    for elemento in risposte_complete[ramo]:
        if not re.search('link', elemento) and not re.search('nodo', elemento):
            if re.search('r\d+', elemento):
                text += '<svgElement style-name="stroke" style-value="#04ed00">simbolo-' + elemento + '</svgElement>'
            elif re.search('g\d+', elemento):
                text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-1</svgElement>'
                text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-2</svgElement>'
                text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-3</svgElement>'
                text += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-4</svgElement>'
    
    for domanda in domande:
        domanda_uppercase = domanda.upper()
        risposte[domanda_uppercase] = text


- Domanda: Quante resistenza ci sono?
- Risposta: Ci sono n resistenze/non c'è alcuna resistenza

In [63]:
domande = [
    '* quante * resistenze *',
]

text = ''
svgElement = '<image>circuiti/circuito.svg</image>'
numResistenze = 0
for nodi in rami:
    for elemento in rami[nodi]:
        if re.search('r\d+', elemento):
            numResistenze += 1
            svgElement += '<svgElement style-name="stroke" style-value="#04ed00">simbolo-' + elemento + '</svgElement>'

if (numResistenze == 0):
    text = 'Non ci sono resistenze'
elif (numResistenze == 1):
    text = 'C\'è una sola resistenza' + text
else:
    text = 'Ci sono ' + str(numResistenze) + ' resistenze' + svgElement

for domanda in domande:
    domanda_uppercase = domanda.upper()
    risposte[domanda_uppercase] = text
    print(domanda_uppercase, text)

* QUANTE * RESISTENZE * Ci sono 6 resistenze<image>circuiti/circuito.svg</image><svgElement style-name="stroke" style-value="#04ed00">simbolo-r1</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-r3</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-r2</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-r4</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-r5</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-r6</svgElement>


- Domanda: Quali sono le resistenze del circuito?
- Risposta: Le resistenze del circuito sono ...

In [64]:
domande = [
    '* quali * resistenze *',
]

numResistenze = 0
text = ''
svgElement = '<image>circuiti/circuito.svg</image>'
for nodi in rami:
    for elemento in rami[nodi]:
        if re.search('r\d+', elemento):
            numResistenze += 1
            text += elemento.upper() + ', '
            svgElement += '<svgElement style-name="stroke" style-value="#04ed00">simbolo-' + elemento + '</svgElement>'

if (numResistenze == 0):
    text = 'Non ci sono resistenze'
elif (numResistenze == 1):
    text = 'C\'è una sola resistenza' + text[:-2] + svgElement
else:
    text = 'Le resistenze del circuito sono ' + text[:-2] + svgElement

for domanda in domande:
    domanda_uppercase = domanda.upper()
    risposte[domanda_uppercase] = text
    print(domanda_uppercase, text)

* QUALI * RESISTENZE * Le resistenze del circuito sono R1, R3, R2, R4, R5, R6<image>circuiti/circuito.svg</image><svgElement style-name="stroke" style-value="#04ed00">simbolo-r1</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-r3</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-r2</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-r4</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-r5</svgElement><svgElement style-name="stroke" style-value="#04ed00">simbolo-r6</svgElement>


- Domanda: Quanti generatori ci sono?
- Risposta: Ci sono n generatori/non c'è alcun generatore

In [65]:
domande = [
    '* quanti * generatori *',
]

text = ''
svgElement = '<image>circuiti/circuito.svg</image>'
numGeneratori = 0
for nodi in rami:
    for elemento in rami[nodi]:
        if re.search('g\d+', elemento):
            numGeneratori += 1
            svgElement += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-1</svgElement>'
            svgElement += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-2</svgElement>'
            svgElement += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-3</svgElement>'
            svgElement += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-4</svgElement>'

if (numGeneratori == 0):
    text = 'Non ci sono generatori'
elif (numGeneratori == 1):
    text = 'C\'è un solo generatore' + svgElement
else:
    text = 'Ci sono ' + str(numGeneratori) + ' generatori' + svgElement

for domanda in domande:
    domanda_uppercase = domanda.upper()
    risposte[domanda_uppercase] = text
    print(domanda_uppercase, text)

* QUANTI * GENERATORI * Ci sono 2 generatori<image>circuiti/circuito.svg</image><svgElement style-name="stroke" style-value="#04ed00">g1-1</svgElement><svgElement style-name="stroke" style-value="#04ed00">g1-2</svgElement><svgElement style-name="stroke" style-value="#04ed00">g1-3</svgElement><svgElement style-name="stroke" style-value="#04ed00">g1-4</svgElement><svgElement style-name="stroke" style-value="#04ed00">g2-1</svgElement><svgElement style-name="stroke" style-value="#04ed00">g2-2</svgElement><svgElement style-name="stroke" style-value="#04ed00">g2-3</svgElement><svgElement style-name="stroke" style-value="#04ed00">g2-4</svgElement>


- Domanda: Quali sono i generatori del circuito?
- Risposta: I generatori del circuito sono ...

In [66]:
domande = [
    '* quali * generatori *',
]

numGeneratori = 0
text = ''
svgElement = '<image>circuiti/circuito.svg</image>'
for nodi in rami:
    for elemento in rami[nodi]:
        if re.search('g\d+', elemento):
            numGeneratori += 1
            text += elemento.upper() + ', '
            svgElement += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-1</svgElement>'
            svgElement += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-2</svgElement>'
            svgElement += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-3</svgElement>'
            svgElement += '<svgElement style-name="stroke" style-value="#04ed00">' + elemento + '-4</svgElement>'

if (numGeneratori == 0):
    text = 'Non ci sono generatori'
elif (numGeneratori == 1):
    text = 'C\'è un solo generatore' + text[:-2] + svgElement
else:
    text = 'I generatori del circuito sono ' + text[:-2] + svgElement

for domanda in domande:
    domanda_uppercase = domanda.upper()
    risposte[domanda_uppercase] = text
    print(domanda_uppercase, text)

* QUALI * GENERATORI * I generatori del circuito sono G1, G2<image>circuiti/circuito.svg</image><svgElement style-name="stroke" style-value="#04ed00">g1-1</svgElement><svgElement style-name="stroke" style-value="#04ed00">g1-2</svgElement><svgElement style-name="stroke" style-value="#04ed00">g1-3</svgElement><svgElement style-name="stroke" style-value="#04ed00">g1-4</svgElement><svgElement style-name="stroke" style-value="#04ed00">g2-1</svgElement><svgElement style-name="stroke" style-value="#04ed00">g2-2</svgElement><svgElement style-name="stroke" style-value="#04ed00">g2-3</svgElement><svgElement style-name="stroke" style-value="#04ed00">g2-4</svgElement>


- Domanda: quanto vale la resistenza ...?
- Risposta: la resitenza vale .../non esiste

In [67]:
for componente, valore in valori_componenti.items():
    domande = [
        '* VALORE * ' + componente + ' *',
        '* VALE * ' + componente + ' *',
    ]

    text = 'Il valore della ' + componente + ' è ' + valore

    if re.search('r\d+', componente):
        text += '<svgElement style-name="stroke" style-value="#04ed00">simbolo-' + componente + '</svgElement>'
    elif re.search('g\d+', componente):
        text += '<svgElement style-name="stroke" style-value="#04ed00">' + componente + '-1</svgElement>'
        text += '<svgElement style-name="stroke" style-value="#04ed00">' + componente + '-2</svgElement>'
        text += '<svgElement style-name="stroke" style-value="#04ed00">' + componente + '-3</svgElement>'
        text += '<svgElement style-name="stroke" style-value="#04ed00">' + componente + '-4</svgElement>'

    text += '<image>circuiti/circuito.svg</image>'


    for domanda in domande:
        domanda_uppercase = domanda.upper()
        risposte[domanda_uppercase] = text
        print(domanda_uppercase, text)


domande = [
    '* VALORE *',
    '* VALE * ',
]

text = 'Il componente non è presente nel circuito'

for domanda in domande:
    domanda_uppercase = domanda.upper()
    risposte[domanda_uppercase] = text
    print(domanda_uppercase, text)

* VALORE * R6 * Il valore della r6 è 3 Ω<svgElement style-name="stroke" style-value="#04ed00">simbolo-r6</svgElement><image>circuiti/circuito.svg</image>
* VALE * R6 * Il valore della r6 è 3 Ω<svgElement style-name="stroke" style-value="#04ed00">simbolo-r6</svgElement><image>circuiti/circuito.svg</image>
* VALORE * R5 * Il valore della r5 è 4 Ω<svgElement style-name="stroke" style-value="#04ed00">simbolo-r5</svgElement><image>circuiti/circuito.svg</image>
* VALE * R5 * Il valore della r5 è 4 Ω<svgElement style-name="stroke" style-value="#04ed00">simbolo-r5</svgElement><image>circuiti/circuito.svg</image>
* VALORE * R4 * Il valore della r4 è 6 Ω<svgElement style-name="stroke" style-value="#04ed00">simbolo-r4</svgElement><image>circuiti/circuito.svg</image>
* VALE * R4 * Il valore della r4 è 6 Ω<svgElement style-name="stroke" style-value="#04ed00">simbolo-r4</svgElement><image>circuiti/circuito.svg</image>
* VALORE * G2 * Il valore della g2 è 6 V<svgElement style-name="stroke" style-valu

## Costruzione del file aiml

In [68]:
import xml.dom.minidom
file_path = 'circuito.aiml'

#crea un nuovo file xml
root = ET.Element('aiml')

for domanda in risposte:
    #crea un tag xml chiamato category
    category = ET.Element('category')
    #inserisci all'interno un altro tag chiaamto pattern contentene un * e crea un tag chiamato template con il valore di text[0]
    pattern = ET.SubElement(category, 'pattern')
    pattern.text = domanda
    template = ET.SubElement(category, 'template')
    template.text = risposte[domanda]
    

    #aggiungi il tag category al tag root
    root.append(category)

#salva il file xml
tree = ET.ElementTree(root)
tree.write(file_path)

# Ottieni la rappresentazione del testo non escapato
xml_str = xml.dom.minidom.parseString(ET.tostring(root)).toprettyxml(indent="    ")

# Sovrascrivi il file AIML con le modifiche
with open(file_path, 'w', encoding='utf-8') as file:
    file.write(xml_str)

In [69]:
import webbrowser

# Apri il file appena creato
webbrowser.open(file_path)

# Leggi il contenuto del file
with open(file_path, 'r', encoding='utf-8') as file:
    file_content = file.read()

# Sostituisci "&lt;" con "<" e "&gt;" con ">"
file_content = file_content.replace("&lt;", "<").replace("&gt;", ">").replace("&quot;", "\"")

# Sovrascrivi il file con le modifiche
with open(file_path, 'w', encoding='utf-8') as file:
    file.write(file_content)